# $p$ - primes


In [ ]:
from sympy import isprime, nextprime, primerange

list(primerange(11, 20))

: 

In [ ]:
ppp = [nextprime(1 << (n + 1)) for n in range(8)][::-1]
print(ppp)

In [ ]:
TEST_P_IDX = 0
TEST_P = ppp[TEST_P_IDX]
TEST_P_IDX, TEST_P

# $V$ - Vandermonde matrices


In [ ]:
import numpy as np

In [ ]:
VVV = []
for p in ppp:
    VVV.append(np.array([[pow(x, j, p) for j in range(p - 1)] for x in range(1, p)]))
VVV

# $A$ - Analysis matrices


In [ ]:
AAA = []

for p in ppp:
    AAA.append(
        np.array([[(-pow(a, -n, p)) % p for n in range(p - 1)] for a in range(1, p)])
    )

AAA

# Matrices Test


In [ ]:
test_vector = np.array([pow(10, i, TEST_P) for i in range(TEST_P - 1)])
t = (AAA[TEST_P_IDX] @ test_vector) % TEST_P
print(t.shape)
t

In [ ]:
test_vector = np.array([pow(10, i, TEST_P) for i in range(TEST_P - 1)]) * 10 % TEST_P
t = (AAA[TEST_P_IDX] @ test_vector) % TEST_P
print(t.shape)
t

In [ ]:
test_vector = (
    np.array([pow(2, i, TEST_P) for i in range(TEST_P - 1)]) * 9
    + np.array([pow(3, i, TEST_P) for i in range(TEST_P - 1)]) * 8
    + np.array([pow(4, i, TEST_P) for i in range(TEST_P - 1)]) * 7
    + np.array([pow(1, i, TEST_P) for i in range(TEST_P - 1)]) * 6
) % TEST_P
t = (AAA[TEST_P_IDX] @ test_vector) % TEST_P
print(t.shape)
t

In [ ]:
np.array_equal(AAA[TEST_P_IDX] @ VVV[TEST_P_IDX].T % TEST_P, np.eye(TEST_P - 1))

In [ ]:
np.array_equal(
    VVV[TEST_P_IDX].T @ (AAA[TEST_P_IDX] @ test_vector) % TEST_P,
    test_vector,
)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(test_vector)

# Error Analysis Test


In [ ]:
test_vector = np.ones((TEST_P - 1,)) * 4
AAA[TEST_P_IDX] @ test_vector % TEST_P

In [ ]:
test_vector[5] += 1

In [ ]:
AAA[TEST_P_IDX] @ test_vector % TEST_P

# Open an existing file


In [ ]:
fs = open("Wildlife.wmv", "rb")
chunk = fs.read(ppp[0] - 1)
fs.close()

In [ ]:
import matplotlib.pyplot as plt

plt.plot(list(chunk))

In [ ]:
AAA[0] @ np.frombuffer(chunk, dtype=np.int8) % ppp[0]

# Adaptivity Test


In [ ]:
test_vector = np.array(list(chunk))
np.array_equal(
    VVV[0].T @ (AAA[0] @ test_vector) % ppp[0],
    test_vector,
)

In [ ]:
test_vector = np.array(list(i & 0b_111_1111 for i in chunk[: ppp[1] - 1]))
np.array_equal(
    VVV[1].T @ (AAA[1] @ test_vector) % ppp[1],
    test_vector,
)

In [ ]:
test_vector = np.array(list(i & 0b_11_1111 for i in chunk[: ppp[2] - 1]))
np.array_equal(
    VVV[2].T @ (AAA[2] @ test_vector) % ppp[2],
    test_vector,
)

In [ ]:
test_vector = np.array(list(i & 0b_1_1111 for i in chunk[: ppp[3] - 1]))
np.array_equal(
    VVV[3].T @ (AAA[3] @ test_vector) % ppp[3],
    test_vector,
)

# Codecs Test


In [ ]:
import struct

F_LITTLE_ENDIAN = "<"
F_UNSIGNED_CHAR = "B"
struct.pack("<B", 0x99)

In [ ]:
b"123" + b"789"

In [ ]:
type(b"")

In [ ]:
import time

## Constant


In [ ]:
from collections import Counter


def encoder_const(bys: bytes) -> list[int]:
    if not bys:
        return []

    mode, mode_count = Counter(bys).most_common(1)[0]

    exceptions = []

    for idx, val in enumerate(bys):
        if val != mode:
            exceptions.extend((idx, val))

    return [
        0x01,  # method = constant
        mode,  # 최빈값
        len(exceptions) // 2,  # 예외 개수
        *exceptions,
    ]


test_str = b"4" * (0x100 - 2) + b"04"

st = time.perf_counter()
t = encoder_const(test_str)
tt = time.perf_counter() - st
print(f"past {tt}; 1/t = {1/tt}")

print(t)
print(" ".join(hex(i) for i in t))
print("encoded ratio :", len(t) / len(test_str))
print("saving rate   :", 1 - len(t) / len(test_str))

In [ ]:
def decoder_const(vals: list[int], n: int) -> bytes:
    """
    Decode the payload produced by ``encoder_const`` after its
    single-byte method header has been removed.

    ``n`` specifies the length of the original byte sequence.
    """
    assert vals, "encoded value must contain a mode"

    mode = vals[0]
    if len(vals) == 1:
        exception_count = 0
        payload_start = 1
    else:
        exception_count = vals[1]
        payload_start = 2

    payload_end = payload_start + 2 * exception_count
    assert len(vals) >= payload_end, "encoded value has incomplete exceptions"

    result = bytearray([mode] * n)
    payload = vals[payload_start:payload_end]

    for idx, value in zip(payload[::2], payload[1::2]):
        result[idx] = value

    return bytes(result)


st = time.perf_counter()
decoded = decoder_const(t[1:], len(test_str))
tt = time.perf_counter() - st
print(f"past {tt}; 1/t = {1/tt}")

assert decoded == test_str
assert decoder_const([ord("4")], len(test_str)) == b"4" * len(test_str)
print(decoded)

## Delta


In [ ]:
def encoder_delta(bys: bytes) -> list[int]:
    assert len(bys) > 1, "input length must be >= 2"
    seed = bys[0]

    deltas = bytes((cur - prev) % 0x100 for prev, cur in zip(bys, bys[1:]))

    encoded_deltas = encoder_const(deltas)

    return [
        0x02,
        seed,
        *encoded_deltas[1:],
    ]


test_str = b"abcdefghijklmnop " * 3

st = time.perf_counter()
t = encoder_delta(test_str)
tt = time.perf_counter() - st
print(f"past {tt}; 1/t = {1/tt}")

print(t)
print(" ".join(hex(i) for i in t))
print("encoded ratio :", len(t) / len(test_str))
print("saving rate   :", 1 - len(t) / len(test_str))

In [ ]:
def decoder_delta(vals: list[int], n: int) -> bytes:
    """
    Decode the payload produced by ``encoder_delta`` after its
    single-byte method header has been removed.

    ``n`` specifies the length of the original byte sequence.
    """

    assert len(vals) >= 2, "encoded value is incomplete"

    seed = vals[0]
    deltas = decoder_const(vals[1:], n - 1)

    result = bytearray([seed])
    for delta in deltas:
        result.append((result[-1] + delta) % 0x100)

    return bytes(result)


st = time.perf_counter()
decoded = decoder_delta(t[1:], len(test_str))
tt = time.perf_counter() - st
print(f"past {tt}; 1/t = {1/tt}")

assert decoded == test_str
print(decoded)

## Delta2


In [ ]:
def encoder_delta2(bys: bytes) -> list[int]:
    assert len(bys) > 2, "input length must be > 2"

    seed0 = bys[0]
    seed1 = bys[1]

    # Δ²x[n] = x[n] - 2*x[n-1] + x[n-2]
    deltas2 = bytes(
        (cur - 2 * prev + prevprev) % 0x100
        for prevprev, prev, cur in zip(bys, bys[1:], bys[2:])
    )

    # 2차 차분 배열에서 가장 흔한 값을 기본값으로 잡고
    # 예외만 저장
    encoded_deltas2 = encoder_const(deltas2)

    return [
        0x03,  # method = delta2
        seed0,  # x[0]
        seed1,  # x[1]
        *encoded_deltas2[1:],  # 내부 constant method 0x01 제거
    ]


test_str = b"abcdefghijklmnop"
test_str += test_str[::-1]

st = time.perf_counter()
t = encoder_delta2(test_str)
tt = time.perf_counter() - st
print(f"past {tt}; 1/t = {1/tt}")

print(t)
print(" ".join(hex(i) for i in t))
print("encoded ratio :", len(t) / len(test_str))
print("saving rate   :", 1 - len(t) / len(test_str))

In [ ]:
def decoder_delta2(vals: list[int], n: int) -> bytes:
    """
    Decode the payload produced by ``encoder_delta2`` after its
    single-byte method header has been removed.

    ``n`` specifies the length of the original byte sequence.
    """
    assert n > 2, "output length must be > 2"
    assert len(vals) > 2, "encoded value is incomplete"

    seed0, seed1 = vals[:2]
    deltas2 = decoder_const(vals[2:], n - 2)

    result = bytearray([seed0, seed1])

    for delta2 in deltas2:
        value = (delta2 + 2 * result[-1] - result[-2]) % 0x100
        result.append(value)

    return bytes(result)


st = time.perf_counter()
decoded = decoder_delta2(t[1:], len(test_str))
tt = time.perf_counter() - st
print(f"past {tt}; 1/t = {1/tt}")

assert decoded == test_str
print(decoded)

## Periodic


In [ ]:
def encoder_periodic(bys: bytes) -> list[int]:
    n = len(bys)

    assert n >= 4, "input length must be >= 4"

    best = None

    # T=1은 delta에서 사실상 더 싸게 잡을 수 있으므로 제외.
    # 진짜 반복 주기로서 최소 2회는 관찰하려고 N//2까지만 탐색.
    for period in range(2, n // 2 + 1):

        exceptions = []

        # 첫 period개는 seed로 저장하므로 그 이후부터 비교.
        for idx in range(period, n):

            # decoder에서도 이미 복원된 idx-period 위치를 참조하게 된다.
            predicted = bys[idx - period]

            if bys[idx] != predicted:
                exceptions.extend((idx, bys[idx]))

        candidate = [
            0x04,  # method = periodic
            period,  # 주기 T
            *bys[:period],  # 첫 T개 seed
            len(exceptions) // 2,  # 예외 개수
            *exceptions,
        ]

        # 예외 개수가 아니라 실제 저장 길이로 경쟁
        if best is None or len(candidate) < len(best):
            best = candidate

    return best


test_str = b"abcd" * 20
test_str += b"0000abcd"

st = time.perf_counter()
t = encoder_periodic(test_str)
tt = time.perf_counter() - st
print(f"past {tt}; 1/t = {1/tt}")

print(t)
print(" ".join(hex(i) for i in t))
print("encoded ratio :", len(t) / len(test_str))
print("saving rate   :", 1 - len(t) / len(test_str))

In [ ]:
def decoder_periodic(vals: list[int], original_len: int) -> bytes:
    # 0x04는 상위 decoder가 이미 제거했다고 가정
    #
    # vals:
    # [period][seed...][exception_count][pos][value]...

    period = vals[0]

    seed_start = 1
    seed_end = seed_start + period

    restored = list(vals[seed_start:seed_end])

    exception_count = vals[seed_end]
    pos = seed_end + 1

    exceptions = {}

    for _ in range(exception_count):
        idx = vals[pos]
        val = vals[pos + 1]

        exceptions[idx] = val
        pos += 2

    # seed 이후 순차 복원
    for idx in range(period, original_len):

        if idx in exceptions:
            restored.append(exceptions[idx])
        else:
            restored.append(restored[idx - period])

    return bytes(restored)


# Periodic codec round-trip checks
st = time.perf_counter()
round_trip = decoder_periodic(t[1:], len(test_str))
tt = time.perf_counter() - st
print(f"past {tt}; 1/t = {1/tt}")
assert round_trip == test_str, (round_trip, test_str)

another = b"abcabcabcabcab"
encoded = encoder_periodic(another)
decoded = decoder_periodic(encoded[1:], len(another))
assert decoded == another, (decoded, another)

print("periodic decoder OK")

## Fourier1


In [ ]:
def encoder_fourier1(bys: bytes, p_idx: int) -> list[int]:
    p = ppp[p_idx]
    n = p - 1

    assert len(bys) == n, f"input length must be {n}"

    x = np.frombuffer(bys, dtype=np.uint8).astype(np.int64)

    best = None
    best_cost = None

    for basis_idx in range(n):
        # basis_idx = a - 1
        #
        # VVV[p_idx][basis_idx][k] = a^k
        #
        # 우리가 찾으려는 모델:
        #
        #     x[k] = c * a^k  (mod p)
        #
        # 각 위치 k가 요구하는 c는
        #
        #     c = x[k] * a^(-k)
        #
        # 이다.
        #
        # 그런데 AAA는
        #
        #     AAA[a-1,k] = -a^(-k)
        #
        # 이므로
        #
        #     c = -x[k] * AAA[a-1,k]
        #
        # 로 바로 계산할 수 있다.

        # p보다 큰 원본 byte는 F_p의 원소로 정확히 예측할 수 없다.
        # 따라서 그런 위치는 무조건 exception이고,
        # coefficient 결정에는 참여시키지 않는다.
        valid = x < p

        if np.any(valid):
            implied_coeffs = (-x[valid] * AAA[p_idx][basis_idx, valid]) % p

            # 가장 많은 위치가 요구하는 coefficient를 선택한다.
            #
            # 이것이 현재 [pos][byte] exception 포맷에서
            # 해당 Fourier basis에 대한 최적 coefficient다.
            values, counts = np.unique(implied_coeffs, return_counts=True)

            coefficient = int(values[np.argmax(counts)])

        else:
            # 이 prime에서 표현 가능한 원본 값이 하나도 없는 경우.
            # 어차피 모든 위치가 exception이 되므로 임의로 0.
            coefficient = 0

        # 선택한 c*a^n으로 전체 블록을 예측한다.
        predicted = (coefficient * VVV[p_idx][basis_idx]) % p

        # 원본 byte와 정확히 같지 않은 곳은 exception.
        mismatch = x != predicted

        exception_count = int(np.count_nonzero(mismatch))

        # 논리적 payload 크기.
        #
        # [05][basis_idx][coeff][count] = 4 symbols
        # exception 하나 = [pos][raw byte] = 2 symbols
        cost = 4 + 2 * exception_count

        # p=257에서 coefficient=256이면
        # 나중에 escape 때문에 최소 1바이트 더 먹는다고 가정.
        # 최종 serializer가 생기면 이 부분은 serializer의 실제 길이로 교체.
        if coefficient == 256:
            cost += 1

        if best is None or cost < best_cost:
            best = (
                basis_idx,
                coefficient,
                mismatch,
            )
            best_cost = cost

    basis_idx, coefficient, mismatch = best

    exceptions = []

    for idx in np.flatnonzero(mismatch):
        exceptions.extend(
            (
                int(idx),
                int(x[idx]),
            )
        )

    return [
        0x05,  # Fourier1
        basis_idx,  # a - 1
        coefficient,  # c
        len(exceptions) // 2,  # exception count
        *exceptions,
    ]


# ---------------------------------------------------------
# Fourier1 test
#
# 이상적인 입력:
#     x[n] = coefficient * basis^n (mod p)
#
# 여기서는
#     p = 17
#     basis = 4
#     coefficient = 7
#
# 로 만든 뒤 두 바이트만 일부러 깨뜨린다.
# ---------------------------------------------------------

p_idx = 0  # TEST_P_IDX
p = ppp[p_idx]
n = p - 1

basis = 4
coefficient = 7

# 완벽한 Fourier1 신호
test_str = bytes((coefficient * pow(basis, i, p)) % p for i in range(n))

# 일부 위치만 고의로 파손
broken = bytearray(test_str)

broken[3] = (broken[3] + 1) % p
broken[11] = (broken[11] + 2) % p

test_str = bytes(broken)

print("test input :", list(test_str))

st = time.perf_counter()
t = encoder_fourier1(test_str, p_idx)
tt = time.perf_counter() - st

print(f"past {tt}; 1/t = {1/tt}")
print(t)
print(" ".join(hex(i) for i in t))
print("encoded ratio :", len(t) / len(test_str))
print("saving rate   :", 1 - len(t) / len(test_str))

In [ ]:
def decoder_fourier1(
    vals: list[int],
    p_idx: int,
) -> bytes:
    # 0x05는 상위 decoder가 이미 제거했다고 가정.
    #
    # vals:
    # [basis_idx][coefficient][exception_count][pos][byte]...

    p = ppp[p_idx]
    n = p - 1

    basis_idx = vals[0]
    coefficient = vals[1]
    exception_count = vals[2]

    # c * a^n으로 기본 블록 복원
    restored = (coefficient * VVV[p_idx][basis_idx]) % p

    restored = restored.astype(np.int64)

    pos = 3

    # 모델과 달랐던 실제 byte만 덮어쓴다.
    for _ in range(exception_count):
        idx = vals[pos]
        raw_byte = vals[pos + 1]

        restored[idx] = raw_byte
        pos += 2

    return bytes(int(x) for x in restored)


st = time.perf_counter()

round_trip = decoder_fourier1(t[1:], p_idx)  # 0x05 method header 제거

tt = time.perf_counter() - st

print(f"past {tt}; 1/t = {1/tt}")

assert round_trip == test_str, (round_trip, test_str)

print("fourier1 decoder OK")
print(list(round_trip))

# Competition


In [ ]:
encoder_candidates = {
    "constant": lambda data: encoder_const(data),
    "delta": lambda data: encoder_delta(data),
    "delta2": lambda data: encoder_delta2(data),
    "periodic": lambda data: encoder_periodic(data),
    "fourier1": lambda data: encoder_fourier1(data, 0),
}

competition_results = []

for name, encoder in encoder_candidates.items():
    try:
        start = time.perf_counter()
        encoded_value = encoder(chunk)
        elapsed = time.perf_counter() - start

        encoded_size = len(encoded_value)
        original_size = len(chunk)

        competition_results.append(
            {
                "encoder": name,
                "size": encoded_size,
                "ratio": encoded_size / original_size,
                "saving_rate": 1 - encoded_size / original_size,
                "time": elapsed,
                "encoded": encoded_value,
            }
        )
    except Exception as exc:
        print(f"{name}: unavailable ({exc})")

competition_results.sort(key=lambda result: result["size"])

for result in competition_results:
    print(
        f"{result['encoder']:>8}: "
        f"{result['size']:4} bytes, "
        f"ratio={result['ratio']:.3f}, "
        f"saving={result['saving_rate']:.1%}, "
        f"time={result['time']:.6f}s"
    )

best_encoder = competition_results[0]
print(f"\nBest encoder: {best_encoder['encoder']}")
print(best_encoder["encoded"])